# 01d — Financial Data Preparation
Loads BTS Form 41 financials for 15 US carriers (2023–2025). Used to derive `profit_margin` and `cost_efficiency` features.
**Source:** [BTS Form 41](https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FIM)
**Output:** `dataset/financials/financials_clean.parquet` — 165 rows (15 carriers × ~11 quarters)

In [1]:
import pandas as pd
import os

dfs = []
folder = 'dataset/financials'
for f in sorted(os.listdir(folder)):
    if f.endswith('.csv'):
        df = pd.read_csv(os.path.join(folder, f))
        dfs.append(df)
        print(f'{f}: {len(df)} rows')

financials = pd.concat(dfs, ignore_index=True)
print(f'\nTotal: {len(financials)} rows')
print(f'Columns: {financials.columns.tolist()}')
print(f'\nCarriers: {financials["UNIQUE_CARRIER"].nunique()}')
print(financials['UNIQUE_CARRIER'].unique())

T_F41SCHEDULE_P12-2.csv: 396 rows
T_F41SCHEDULE_P12-3.csv: 292 rows
T_F41SCHEDULE_P12.csv: 419 rows

Total: 1107 rows
Columns: ['NET_INCOME', 'OP_PROFIT_LOSS', 'TRANS_REV_PAX', 'OP_REVENUES', 'FLYING_OPS', 'MAINTENANCE', 'PAX_SERVICE', 'DEPREC_AMORT', 'TRANS_EXPENSES', 'OP_EXPENSES', 'UNIQUE_CARRIER', 'UNIQUE_CARRIER_NAME', 'REGION', 'YEAR', 'QUARTER']

Carriers: 56
['FX' 'B6' 'NK' 'AA' 'WN' '5X' 'HA' 'AS' 'UA' '9E' 'DL' 'MX' '5Y' 'YV'
 'OH' '3M' 'SY' 'N8' 'KD' 'XP' 'NC' 'F9' 'L2' 'MQ' 'G4' 'X9' 'ZW' '1BQ'
 'M6' 'GCA' '37Q' 'ABX' 'U7' 'KLQ' '8C' '5V' 'KAQ' 'PT' 'WL' 'GFQ' 'G7'
 '3EQ' 'PO' 'QX' 'YX' 'OO' 'C5' 'PFQ' '09Q' '27Q' '2PQ' 'WI' 'KH' '1EQ'
 '3FQ' '1TQ']


In [2]:
my_airlines = ['AA','DL','UA','WN','B6','AS','NK','F9','G4','HA',
               'MQ','OO','YX','OH','9E']

fin_clean = financials[financials['UNIQUE_CARRIER'].isin(my_airlines)].copy()
print(f"Filtered: {len(fin_clean)} rows")
print(f"Airlines: {fin_clean['UNIQUE_CARRIER'].nunique()}")
print(fin_clean.groupby(['UNIQUE_CARRIER','YEAR'])['QUARTER'].count().unstack().to_string())

Filtered: 396 rows
Airlines: 15
YEAR            2023  2024  2025
UNIQUE_CARRIER                  
9E                 8     8     6
AA                16    16    12
AS                 8     8     6
B6                12    12     9
DL                16    16    12
F9                 8     8     6
G4                 4     4     3
HA                 8     8     6
MQ                 8     8     6
NK                 8     8     6
OH                 8     8     6
OO                 8     8     6
UA                16    16    12
WN                 8     8     6
YX                 8     8     6


In [3]:
print(fin_clean[fin_clean['UNIQUE_CARRIER']=='AA'][['UNIQUE_CARRIER','YEAR','QUARTER','REGION','OP_REVENUES']].head(20).to_string())

    UNIQUE_CARRIER  YEAR  QUARTER REGION  OP_REVENUES
7               AA  2024        1      A   1214276.18
22              AA  2024        3      A   2439491.69
24              AA  2024        1      P    365061.01
30              AA  2024        3      P    355594.72
32              AA  2024        4      P    448091.59
37              AA  2024        4      A   1598827.67
52              AA  2024        2      P    333494.73
296             AA  2024        3      D   9267740.68
335             AA  2024        2      A   2329272.10
348             AA  2024        1      D   8873504.56
360             AA  2024        3      L   1581962.45
367             AA  2024        2      L   1725241.55
368             AA  2024        1      L   2116030.29
369             AA  2024        4      L   1882228.88
386             AA  2024        2      D   9944815.08
387             AA  2024        4      D   9728695.54
401             AA  2025        1      A   1186950.28
409             AA  2025    

In [4]:
fin_domestic = fin_clean[fin_clean['REGION'] == 'D'].copy()
print(f"Domestic only: {len(fin_domestic)} rows")
print(fin_domestic.groupby(['UNIQUE_CARRIER','YEAR'])['QUARTER'].count().unstack().to_string())

Domestic only: 165 rows
YEAR            2023  2024  2025
UNIQUE_CARRIER                  
9E                 4     4     3
AA                 4     4     3
AS                 4     4     3
B6                 4     4     3
DL                 4     4     3
F9                 4     4     3
G4                 4     4     3
HA                 4     4     3
MQ                 4     4     3
NK                 4     4     3
OH                 4     4     3
OO                 4     4     3
UA                 4     4     3
WN                 4     4     3
YX                 4     4     3


In [5]:
fin_domestic.to_parquet('dataset/financials/financials_clean.parquet', index=False)
print(f"Saved: {len(fin_domestic)} rows")

Saved: 165 rows


In [6]:
df=pd.read_parquet("/Users/harshithnr/flight_delay_predictions/dataset/financials/financials_clean.parquet")

In [7]:
df.head()

,NET_INCOME,OP_PROFIT_LOSS,TRANS_REV_PAX,OP_REVENUES,FLYING_OPS,MAINTENANCE,PAX_SERVICE,DEPREC_AMORT,TRANS_EXPENSES,OP_EXPENSES,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,REGION,YEAR,QUARTER
0,-539748.67,-117804.23,1333272.28,1558333.44,799886.03,157381.39,159208.60,114840.58,392.83,1676137.68,B6,JetBlue Airways,D,2024,1
1,-520092.14,-398791.62,543730.03,1038862.98,611263.80,75788.40,74522.95,66966.33,0.00,1437654.60,NK,Spirit Air Lines,D,2024,4
2,-406248.43,-500721.12,1259141.55,1501303.86,927668.65,202931.39,193397.61,140276.33,414.51,2002024.98,B6,JetBlue Airways,D,2024,3
3,-280406.58,-268573.79,557033.48,1084576.31,702249.35,81780.66,79446.82,76129.56,0.00,1353150.09,NK,Spirit Air Lines,D,2024,3
4,-223225.00,-379934.00,5263468.00,6123075.00,2713187.00,632386.00,513802.00,398286.00,32917.00,6503009.00,WN,Southwest Airlines Co.,D,2024,1


In [8]:
print(f"Shape: {fin_domestic.shape}")
print(f"\nColumns: {fin_domestic.columns.tolist()}")
print(f"\nNulls:\n{fin_domestic.isnull().sum()}")
print(f"\nSample:")
print(fin_domestic[['UNIQUE_CARRIER','YEAR','QUARTER','OP_REVENUES','OP_EXPENSES','NET_INCOME','MAINTENANCE']].head(10).to_string())

Shape: (165, 15)

Columns: ['NET_INCOME', 'OP_PROFIT_LOSS', 'TRANS_REV_PAX', 'OP_REVENUES', 'FLYING_OPS', 'MAINTENANCE', 'PAX_SERVICE', 'DEPREC_AMORT', 'TRANS_EXPENSES', 'OP_EXPENSES', 'UNIQUE_CARRIER', 'UNIQUE_CARRIER_NAME', 'REGION', 'YEAR', 'QUARTER']

Nulls:
NET_INCOME             0
OP_PROFIT_LOSS         0
TRANS_REV_PAX          0
OP_REVENUES            0
FLYING_OPS             0
MAINTENANCE            0
PAX_SERVICE            0
DEPREC_AMORT           0
TRANS_EXPENSES         0
OP_EXPENSES            0
UNIQUE_CARRIER         0
UNIQUE_CARRIER_NAME    0
REGION                 0
YEAR                   0
QUARTER                0
dtype: int64

Sample:
   UNIQUE_CARRIER  YEAR  QUARTER  OP_REVENUES  OP_EXPENSES  NET_INCOME  MAINTENANCE
3              B6  2024        1   1558333.44   1676137.68  -539748.67    157381.39
4              NK  2024        4   1038862.98   1437654.60  -520092.14     75788.40
6              B6  2024        3   1501303.86   2002024.98  -406248.43    202931.39
8   